# PolyWhisper — Whisper Baselines (Small/Medium/Large)

**Resumable**: if Colab disconnects, re-run this notebook — it skips completed work via state file on HF Hub.

**Runtime**: ~30 min (small), ~45 min (medium), ~1h (large) on T4 GPU.

**Output**: per-language WER/CER + per-sample JSONs → uploaded to `eulogik/polywhisper` on HF Hub.

In [ ]:
#@title 1. Setup — install deps & clone repo
!pip install -q datasets transformers torch soundfile huggingface_hub torchaudio

import os, sys
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    from huggingface_hub import notebook_login
    notebook_login()
    HF_TOKEN = os.environ.get("HF_TOKEN", "")

# Clone repo for the script
if not os.path.exists("PolyWhisper"):
    !git clone -q https://github.com/eulogik/PolyWhisper.git
    !git -C PolyWhisper pull -q
else:
    !git -C PolyWhisper pull -q

sys.path.insert(0, "PolyWhisper")
print("Setup complete")

In [ ]:
#@title 2. Run baselines — pick size below

#@markdown Select which whisper size to evaluate:
size = "small" #@param ["small", "medium", "large"]

#@markdown Languages to evaluate (comma-separated):
langs = "hi,ta,te,bn,mr" #@param {type:"string"}

lang_list = [l.strip() for l in langs.split(",")]

#@markdown Or run ALL sizes sequentially (overrides size above):
run_all_sizes = False #@param {type:"boolean"}

if run_all_sizes:
    for s in ["small", "medium", "large"]:
        print(f"\n{'#'*60}")
        print(f"# BASELINES: {s}")
        print(f"{'#'*60}")
        !cd PolyWhisper && HF_TOKEN={HF_TOKEN} python baselines_resumable.py --repo eulogik/polywhisper --size {s} --langs {langs}
else:
    !cd PolyWhisper && HF_TOKEN={HF_TOKEN} python baselines_resumable.py --repo eulogik/polywhisper --size {size} --langs {langs}

In [ ]:
#@title 3. Check state — see what's completed
from huggingface_hub import hf_hub_download
import json

repo = "eulogik/polywhisper"
try:
    path = hf_hub_download(repo_id=repo, filename="baselines_state.json", repo_type="model")
    state = json.load(open(path))
    print(f"Last update: {state['last_update']}")
    print(f"Completed ({len(state['completed'])}):")
    for k, v in sorted(state["completed"].items()):
        print(f"  {k}: WER {v['wer']:.1f}% CER {v['cer']:.1f}% ({v['n']} samples, {v['time_s']}s)")
except Exception as e:
    print(f"No state found: {e}")

In [ ]:
#@title 4. Generate paper table
from huggingface_hub import hf_hub_download
import json

repo = "eulogik/polywhisper"
try:
    path = hf_hub_download(repo_id=repo, filename="baselines_state.json", repo_type="model")
    state = json.load(open(path))
except:
    print("No state found — run baselines first"); exit()

# Load our expert results
path2 = hf_hub_download(repo_id=repo, filename="fleurs_normalized_results.json", repo_type="model")
experts = json.load(open(path2))

print("| Lang | Expert WER/CER | ", end="")
sizes_done = sorted(set(k.split("_")[0] for k in state["completed"]))
for s in sizes_done:
    print(f"{s} WER/CER | ", end="")
print("script-match |")
print("|---|---|", end="")
for s in sizes_done:
    print("---|", end="")
print("---|")

for lang in ["hi", "ta", "te", "bn", "mr"]:
    e = experts.get(lang, {})
    e_pure = e.get("pure", {})
    sm = e_pure.get("script_matched", {})
    e_str = f"{sm.get('wer', 0):.1f} / {sm.get('cer', 0):.1f}"
    line = f"| {lang} | {e_str} | "
    for s in sizes_done:
        key = f"{s}_{lang}"
        if key in state["completed"]:
            c = state["completed"][key]
            line += f"{c['wer']:.1f} / {c['cer']:.1f} | "
        else:
            line += "— | "
    line += f"{sm.get('rate', 0):.1f}% |"
    print(line)